# Notebook 10 — Cheminformatics Workflows: Virtual Screening, Clustering, and Diversity Analysis

This **capstone notebook** combines everything from the previous notebooks into end-to-end workflows used in real drug discovery programs.

We will work through a complete cheminformatics pipeline on a set of approved kinase inhibitors — the same class of drugs that revolutionized targeted cancer therapy starting with Imatinib (Gleevec) in 2001. Along the way, you will see how the individual techniques we have covered (fingerprints, descriptors, substructure search, filtering) fit together into coherent workflows that computational chemists use daily.

**What we will cover:**

| Workflow | What It Does | Where You Have Seen It |
|----------|-------------|----------------------|
| Scaffold analysis | Extracts molecular frameworks | Notebooks 3-4 (substructure) |
| Chemical space visualization | Projects fingerprints to 2D | Notebook 5 (fingerprints) |
| Clustering | Groups similar compounds | Notebook 5 (similarity) |
| Diversity picking | Selects maximally diverse subsets | Notebook 5 (fingerprints) |
| Virtual screening pipeline | End-to-end hit finding | Notebooks 5-8 combined |
| R-group decomposition | Computational SAR tables | Notebook 4 (substructure) |

> **Wet-lab bridge:** Think of this notebook as the computational equivalent of planning a screening campaign — you would not just throw every compound at an assay. You would curate your library, understand its diversity, cluster it into series, and pick representatives. That is exactly what we are doing here, just faster.

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator, FilterCatalog
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdRGroupDecomposition
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit.ML.Cluster import Butina
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# Plotting defaults
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")
print("All imports successful.")

## 1. Embedded Kinase Inhibitor Dataset

We embed ~30 approved kinase inhibitors directly as SMILES — no downloads needed. These drugs span multiple kinase families (BCR-ABL, EGFR, VEGFR, ALK, JAK, BTK) and represent several generations of kinase inhibitor design.

> **Wet-lab context:** If you have worked in oncology, you have likely used several of these drugs or their analogs in cell-based assays. Imatinib, the "poster child" of targeted therapy, inhibits BCR-ABL. Gefitinib and Erlotinib target EGFR. Crizotinib targets ALK. Despite targeting different kinases, many of these drugs share a common pharmacophore: an aromatic hinge-binding motif that mimics the adenine ring of ATP. This structural commonality is exactly what scaffold analysis will reveal.

In [ ]:
# ── Embedded kinase inhibitor library ──────────────────────────────────
KINASE_INHIBITORS = [
    ("Imatinib",      "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"),
    ("Dasatinib",     "CC1=C(C(=O)N(C=N1)C2=CC(=CC=C2)NC(=O)C3=CC(=NC=C3)NC4=CC(=C(C=C4)Cl)OC5=CC=CC=C5)C"),
    ("Nilotinib",     "CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)NC3=NC=CC(=N3)C4=CN=CC=C4)C(F)(F)F)NC5=NC=CC=N5"),
    ("Gefitinib",     "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4"),
    ("Erlotinib",     "COC=1C=C2C(=NC=NC2=CC1OCCCOC)NC3=CC(=CC=C3)C#C"),
    ("Lapatinib",     "CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3NC4=CC(=C(C=C4)Cl)OCC5=CC(=CC=C5)F"),
    ("Sorafenib",     "CNC(=O)C1=CC(=CC=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC(=C(C=C3)Cl)C(F)(F)F"),
    ("Sunitinib",     "CCN(CC)CCNC(=O)C1=C(C(=C(/C1=C\\2/C3=CC=CC=C3NC2=O)C)C)F"),
    ("Pazopanib",     "CC1=C(C=C(C=C1)NC2=NC=C(C(=N2)N)N3CCN(CC3)C)S(=O)(=O)N"),
    ("Vandetanib",    "CC1=C2C=C(C=CC2=NC=C1COC3=CC=C(C=C3)CN4CCNCC4)OC"),
    ("Crizotinib",    "CC(C1=C(C=CC(=C1)F)NC2=NC=C(C=C2)OC3=CN(N=C3)C4CCNCC4)O"),
    ("Vemurafenib",   "CCCS(=O)(=O)NC1=CC(=C(C=C1F)C(=O)C2=CNC3=CC=C(C=C23)C4=CC=C(C=C4)Cl)F"),
    ("Dabrafenib",    "CC(C)(C)C1=NC(=C(S1)C2=CC=NC(=C2)NC3=C(C=C(C=C3)F)F)C4=CC(=CC=C4)NS(=O)(=O)C(F)(F)F"),
    ("Trametinib",    "CC1=CC2=C(C=C1)N(C(=O)C3=C(N2)C=CC(=C3)NC(=O)C4=CC(=CC=C4F)F)C5CC(O)C(C5)O"),
    ("Ibrutinib",     "C=CC(=O)N1CCC(CC1)N2C3=CC=CC=C3N=C2C4=CC=C(C=C4)OC5=CC=CC=C5"),
    ("Ruxolitinib",   "N#CCC(C1CCCC1)N1CC2=C(C=C(N=C2)C2=CC=NC=C2)C1=O"),
    ("Tofacitinib",   "CC1CCN(CC1NC(=O)C2=C(C=CC(=C2)N3CCOCC3)C#N)C(=O)C4CC4"),
    ("Axitinib",      "CNC(=O)C1=CC=CC=C1/C=C/C2=CC3=CC=C(C=C3N2)S(=O)(=O)C"),
    ("Cabozantinib",  "COC1=CC2=C(C=C1OC)C(=CC=N2)OC3=CC=C(C=C3)NC(=O)C4(CC4)C(=O)NC5=CC=C(C=C5)F"),
    ("Lenvatinib",    "COC1=C(C=C2C(=C1)C(=CC=N2)OC3=CC=C(C=C3)NC(=O)NC4CC4)C(=O)N"),
    ("Regorafenib",   "CNC(=O)C1=CC(=CC=C1)OC2=CC(=C(C=C2)NC(=O)NC3=CC(=C(C=C3)Cl)C(F)(F)F)F"),
    ("Ponatinib",     "CC1=C(C=CC(=C1)C(=O)NC2=CC(=CC=C2)C#CC3=CN=C4C=CC(=NN34)N)C#CC5=CC=C(C=C5)CN6CCN(CC6)C"),
    ("Osimertinib",   "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)NC(=O)/C=C/CN(C)C"),
    ("Alectinib",     "CCC1=CC2=C(C=C1N)N(C(=O)C3=CC=CC=C32)C4=CC=C(C=C4)C#N"),
    ("Ceritinib",     "CC(C)OC1=C(C=C(C(=C1)Cl)NC2=NC=C(C(=N2)NC3=CC=CC=C3S(=O)(=O)C(C)C)C3CCNCC3)OC"),
    ("Bosutinib",     "COC1=CC(=C(C=C1Cl)NC2=C(C=C(C=C2C#N)OC3=CC=C(C=C3)CNCCN4CCOCC4)Cl)OC"),
    ("Afatinib",      "CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OC4CCOC4"),
    ("Brigatinib",    "CC(C)(C1=CC(=CC(=C1)NC(=O)C2=CC=C(C=C2)C(=O)N3CCN(CC3)C)N)P(=O)(C)C"),
    ("Lorlatinib",    "C1CC1NC(=O)C1=CC(=C(N=C1)C1=C(C=CC=C1F)NC(=O)C1=CC=C(N1)C#N)OC"),
]

# Build DataFrame
df = pd.DataFrame(KINASE_INHIBITORS, columns=["Name", "SMILES"])
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)

# Report any parsing failures
n_before = len(df)
df = df.dropna(subset=["mol"]).reset_index(drop=True)
n_after = len(df)
if n_before != n_after:
    print(f"Warning: {n_before - n_after} SMILES failed to parse")
print(f"Loaded {n_after} kinase inhibitors into library")
df[["Name", "SMILES"]].head(10)

In [ ]:
# Quick visual check — show first 12 molecules as a grid
Draw.MolsToGridImage(
    df["mol"].tolist()[:12],
    molsPerRow=4,
    subImgSize=(350, 250),
    legends=df["Name"].tolist()[:12],
)

## 2. Scaffold Analysis

### Chemistry refresher: Murcko Scaffolds

In medicinal chemistry, the **scaffold** (or **framework**) of a molecule is its core ring system plus the linkers connecting the rings. Side chains, substituents, and pendant groups are stripped away.

**Murcko decomposition** (Bemis & Murcko, 1996) formalizes this:
- **Murcko scaffold** = ring systems + linker chains (retains heteroatoms, bond orders)
- **Generic scaffold** = further abstraction where all atoms become carbon, all bonds become single

This is the computational analog of what medicinal chemists do instinctively when they look at a compound series and say *"these all share the same core."*

**Scaffold hopping** — finding a new framework that retains activity — is one of the holy grails of med chem. Computationally, it means finding molecules with *different* scaffolds but *similar* activity profiles.

> **Wet-lab bridge:** When you sketch SAR tables on a whiteboard, you are implicitly doing Murcko decomposition — drawing a core and listing R-groups. RDKit just automates it.

In [ ]:
# ── Extract Murcko scaffolds ──────────────────────────────────────────
df["scaffold"] = df["mol"].apply(lambda m: MurckoScaffold.GetScaffoldForMol(m))
df["scaffold_smi"] = df["scaffold"].apply(Chem.MolToSmiles)

df["generic_scaffold"] = df["mol"].apply(
    lambda m: MurckoScaffold.MakeScaffoldGeneric(MurckoScaffold.GetScaffoldForMol(m))
)
df["generic_smi"] = df["generic_scaffold"].apply(Chem.MolToSmiles)

# ── Count unique scaffolds ────────────────────────────────────────────
print(f"Total molecules:           {len(df)}")
print(f"Unique Murcko scaffolds:   {df['scaffold_smi'].nunique()}")
print(f"Unique generic scaffolds:  {df['generic_smi'].nunique()}")
print(f"\nScaffold diversity ratio:  {df['scaffold_smi'].nunique() / len(df):.2f}")
print("  (1.0 = every molecule has a unique scaffold; low = many share cores)\n")

# ── Most common scaffolds ─────────────────────────────────────────────
print("Most common Murcko scaffolds:")
scaffold_counts = df["scaffold_smi"].value_counts()
for smi, count in scaffold_counts.head(5).items():
    members = df[df["scaffold_smi"] == smi]["Name"].tolist()
    print(f"  [{count}] {smi[:60]}{'...' if len(smi) > 60 else ''}")
    print(f"       Members: {', '.join(members)}")

In [ ]:
# ── Visualize top scaffolds ───────────────────────────────────────────
# Show the most common scaffolds as a grid
top_scaffolds = scaffold_counts.head(6)
scaffold_mols = [Chem.MolFromSmiles(smi) for smi in top_scaffolds.index]
scaffold_labels = [f"n={count}" for count in top_scaffolds.values]

Draw.MolsToGridImage(
    scaffold_mols,
    molsPerRow=3,
    subImgSize=(350, 250),
    legends=scaffold_labels,
)

## 3. Chemical Space Visualization

### Chemistry refresher: Why Dimensionality Reduction?

A Morgan fingerprint with 2048 bits lives in a 2048-dimensional space. Our brains cannot visualize that, so we use **dimensionality reduction** to project it into 2D while preserving as much structure as possible.

Two common methods:

| Method | How it works | Strengths | Weaknesses |
|--------|-------------|-----------|------------|
| **PCA** | Linear projection maximizing variance | Fast, deterministic, preserves global structure | Misses nonlinear relationships |
| **t-SNE** | Nonlinear embedding preserving local neighborhoods | Reveals clusters beautifully | Stochastic, distances between clusters are meaningless |

> **Wet-lab bridge:** This is like comparing TLC plates (quick, linear separation) vs. 2D-NMR (rich local detail but harder to interpret globally). PCA gives you the TLC view; t-SNE gives you the NOESY view of chemical space.

**Important caveats:**
- Proximity in these plots suggests similarity, but distances are not quantitative
- t-SNE cluster shapes/sizes do not mean anything — only membership matters
- Always use fingerprint-based similarity for quantitative comparisons

In [ ]:
# ── Generate fingerprints and convert to numpy array ──────────────────
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [morgan_gen.GetFingerprint(mol) for mol in df["mol"]]

# Convert to dense numpy array for sklearn
fp_array = np.zeros((len(fps), 2048), dtype=np.float32)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, fp_array[i])

print(f"Fingerprint matrix shape: {fp_array.shape}")
print(f"Average bits set per molecule: {fp_array.sum(axis=1).mean():.0f} / 2048")

In [ ]:
# ── PCA and t-SNE side by side ─────────────────────────────────────────
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(fp_array)

tsne = TSNE(n_components=2, random_state=42, perplexity=min(8, len(df) - 1))
tsne_coords = tsne.fit_transform(fp_array)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA plot
axes[0].scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    alpha=0.7, edgecolors="black", linewidth=0.5, s=60, c="steelblue"
)
for i, name in enumerate(df["Name"]):
    axes[0].annotate(
        name, (pca_coords[i, 0], pca_coords[i, 1]),
        fontsize=6, alpha=0.7, ha="center", va="bottom",
        xytext=(0, 5), textcoords="offset points"
    )
axes[0].set_title("PCA of Morgan Fingerprints", fontsize=13)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")

# t-SNE plot
axes[1].scatter(
    tsne_coords[:, 0], tsne_coords[:, 1],
    alpha=0.7, edgecolors="black", linewidth=0.5, s=60, c="coral"
)
for i, name in enumerate(df["Name"]):
    axes[1].annotate(
        name, (tsne_coords[i, 0], tsne_coords[i, 1]),
        fontsize=6, alpha=0.7, ha="center", va="bottom",
        xytext=(0, 5), textcoords="offset points"
    )
axes[1].set_title("t-SNE of Morgan Fingerprints", fontsize=13)
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

print(f"\nPCA: first 2 components explain {pca.explained_variance_ratio_[:2].sum():.1%} of total variance")

## 4. Clustering

### Chemistry refresher: Why Cluster Compounds?

Clustering groups structurally similar compounds together. This is essential for:

1. **Identifying compound series** — which molecules belong to the same chemical family?
2. **Selecting representatives** — pick one from each cluster for a focused screen
3. **Understanding SAR** — within a cluster, activity differences reveal key substituent effects

Two approaches we will use:

**Butina clustering** (Taylor, 1995):
- Uses Tanimoto distance directly (no conversion to Euclidean space needed)
- Produces clusters of variable size based on a distance cutoff
- Fast, deterministic, and well-suited for chemical data
- A cutoff of 0.6 distance (= 0.4 Tanimoto similarity) typically produces chemically meaningful clusters

**KMeans clustering**:
- Requires a fixed number of clusters *k* (you must choose)
- Works on the numpy fingerprint array
- Clusters are always convex in feature space
- Useful when you need exactly *k* groups (e.g., "give me 5 series")

> **Wet-lab bridge:** Clustering is like sorting your compound plate by structural series before running an assay — you want to know which hits are related (same series) vs. independent (different scaffolds). A cluster of active compounds = a tractable SAR series. Singletons = potential novel leads worth following up.

In [ ]:
# ── Butina clustering ─────────────────────────────────────────────────
# Step 1: Compute pairwise Tanimoto distance matrix (lower triangle)
n = len(fps)
dists = []
for i in range(1, n):
    sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    dists.extend([1 - s for s in sims])

# Step 2: Cluster with distance cutoff
# 0.6 distance = compounds with Tanimoto >= 0.4 can be in the same cluster
cutoff = 0.6
clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)

print(f"Butina clustering (cutoff={cutoff}):")
print(f"  Number of clusters: {len(clusters)}")
print(f"  Largest cluster:    {len(clusters[0])} compounds")
print(f"  Singletons:         {sum(1 for c in clusters if len(c) == 1)}")
print()

# Assign cluster IDs to DataFrame
cluster_id_map = {}
for cid, cluster in enumerate(clusters):
    for idx in cluster:
        cluster_id_map[idx] = cid
df["butina_cluster"] = [cluster_id_map[i] for i in range(len(df))]

# Print cluster membership
for i, cluster in enumerate(clusters):
    names = [df.iloc[idx]["Name"] for idx in cluster]
    print(f"  Cluster {i} ({len(cluster)}): {', '.join(names)}")

In [ ]:
# ── KMeans clustering ─────────────────────────────────────────────────
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["kmeans_cluster"] = kmeans.fit_predict(fp_array)

print("KMeans clustering (k=5):")
for k in range(5):
    members = df[df["kmeans_cluster"] == k]["Name"].tolist()
    print(f"  Cluster {k} ({len(members)}): {', '.join(members)}")

In [ ]:
# ── Visualize KMeans clusters on PCA ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Butina clusters on PCA
n_butina = df["butina_cluster"].nunique()
cmap_butina = plt.cm.get_cmap("tab20", n_butina)
scatter1 = axes[0].scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    c=df["butina_cluster"], cmap=cmap_butina,
    alpha=0.8, edgecolors="black", linewidth=0.5, s=70
)
for i, name in enumerate(df["Name"]):
    axes[0].annotate(
        name, (pca_coords[i, 0], pca_coords[i, 1]),
        fontsize=5.5, alpha=0.7, ha="center", va="bottom",
        xytext=(0, 5), textcoords="offset points"
    )
axes[0].set_title(f"Butina Clusters (cutoff={cutoff}, n={n_butina})", fontsize=12)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# KMeans clusters on PCA
scatter2 = axes[1].scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    c=df["kmeans_cluster"], cmap="tab10",
    alpha=0.8, edgecolors="black", linewidth=0.5, s=70
)
for i, name in enumerate(df["Name"]):
    axes[1].annotate(
        name, (pca_coords[i, 0], pca_coords[i, 1]),
        fontsize=5.5, alpha=0.7, ha="center", va="bottom",
        xytext=(0, 5), textcoords="offset points"
    )
axes[1].legend(*scatter2.legend_elements(), title="Cluster", fontsize=8)
axes[1].set_title("KMeans Clusters (k=5)", fontsize=12)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.show()

## 5. Diversity Picking

### Chemistry refresher: Why Diversity Matters

When resources are limited — synthesis capacity, screening slots, animal studies — you cannot test everything. **Diversity picking** maximizes the information gained per compound tested by selecting the most structurally diverse subset.

**MaxMinPicker** (RDKit's implementation):
1. Start with a seed compound (or random)
2. Greedily pick the compound that **maximizes** the **minimum** distance to any already-selected compound
3. Repeat until you have *n* picks

This ensures the picked set is spread evenly across chemical space — no two picks are too similar.

> **Wet-lab bridge:** This is exactly the logic behind cherry-picking compounds for a focused screen. If you have 1000 HTS hits but can only retest 50, you do not want 50 analogs of the same scaffold. You want representatives spanning all the structural series. MaxMinPicker automates this decision.

**Practical tip:** Diversity picking from *filtered* hits (post-Lipinski, post-PAINS) is more useful than picking from the raw library. Always filter first, then diversify.

In [ ]:
# ── MaxMinPicker — select diverse subset ──────────────────────────────
picker = rdSimDivPickers.MaxMinPicker()

n_to_pick = 10
pick_indices = list(picker.LazyBitVectorPick(fps, len(fps), n_to_pick))
picked_names = [df.iloc[i]["Name"] for i in pick_indices]

print(f"MaxMin diversity pick ({n_to_pick} from {len(df)}):")
for rank, (idx, name) in enumerate(zip(pick_indices, picked_names), 1):
    print(f"  {rank:2d}. {name}")

# Verify diversity: compute pairwise similarity within picked set
picked_fps = [fps[i] for i in pick_indices]
picked_sims = []
for i in range(len(picked_fps)):
    for j in range(i + 1, len(picked_fps)):
        picked_sims.append(DataStructs.TanimotoSimilarity(picked_fps[i], picked_fps[j]))

print(f"\nPairwise Tanimoto within picked set:")
print(f"  Mean: {np.mean(picked_sims):.3f}")
print(f"  Max:  {np.max(picked_sims):.3f}")
print(f"  Min:  {np.min(picked_sims):.3f}")

# Compare to random pick
rng = np.random.default_rng(42)
random_indices = rng.choice(len(df), size=n_to_pick, replace=False)
random_fps = [fps[i] for i in random_indices]
random_sims = []
for i in range(len(random_fps)):
    for j in range(i + 1, len(random_fps)):
        random_sims.append(DataStructs.TanimotoSimilarity(random_fps[i], random_fps[j]))

print(f"\nPairwise Tanimoto within random set (for comparison):")
print(f"  Mean: {np.mean(random_sims):.3f}")
print(f"  Max:  {np.max(random_sims):.3f}")
print(f"  → MaxMin picks are more diverse (lower mean similarity)")

In [ ]:
# ── Visualize diversity picks on PCA ──────────────────────────────────
picked_set = set(pick_indices)
colors = ["#e74c3c" if i in picked_set else "#bdc3c7" for i in range(len(df))]
sizes = [120 if i in picked_set else 40 for i in range(len(df))]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    c=colors, s=sizes, alpha=0.85, edgecolors="black", linewidth=0.5
)

# Label only picked compounds
for i, name in enumerate(df["Name"]):
    if i in picked_set:
        ax.annotate(
            name, (pca_coords[i, 0], pca_coords[i, 1]),
            fontsize=7, fontweight="bold", ha="center", va="bottom",
            xytext=(0, 7), textcoords="offset points",
            bbox=dict(boxstyle="round,pad=0.2", fc="yellow", alpha=0.6)
        )

ax.set_title(f"MaxMin Diversity Pick (n={n_to_pick}, red) vs. Full Library (gray)", fontsize=13)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

## 6. Virtual Screening Pipeline

### End-to-end workflow combining everything

This section demonstrates a complete virtual screening workflow — the kind of pipeline that runs in real drug discovery programs. We will:

1. **Define a query** — start from a known active (Imatinib)
2. **Similarity search** — find analogs in our library
3. **Property filter** — apply Lipinski's Rule of Five
4. **Diversity pick** — select diverse representatives from hits

> **Wet-lab bridge:** This mirrors the real-world process: you have a hit from an HTS, you search your compound collection for analogs, filter for drug-likeness, then cherry-pick a diverse set for dose-response confirmation. The computational version runs in seconds instead of days.

Each step progressively narrows the funnel — just like a real screening cascade.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# VIRTUAL SCREENING PIPELINE
# ══════════════════════════════════════════════════════════════════════

# ── Step 1: Define query molecule ─────────────────────────────────────
query_name = "Imatinib"
query_smi = "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"
query = Chem.MolFromSmiles(query_smi)
query_fp = morgan_gen.GetFingerprint(query)

print(f"Query: {query_name}")
print(f"MW: {Descriptors.MolWt(query):.1f}, LogP: {Descriptors.MolLogP(query):.2f}")
print(f"HBD: {rdMolDescriptors.CalcNumHBD(query)}, HBA: {rdMolDescriptors.CalcNumHBA(query)}")
Draw.MolToImage(query, size=(400, 300))

In [ ]:
# ── Step 2: Similarity search ─────────────────────────────────────────
df["similarity_to_query"] = df["mol"].apply(
    lambda m: DataStructs.TanimotoSimilarity(query_fp, morgan_gen.GetFingerprint(m))
)

# Sort by similarity (excluding query itself if present)
df_ranked = df.sort_values("similarity_to_query", ascending=False)
print("Step 1 — Similarity ranking (all compounds):")
print(df_ranked[["Name", "similarity_to_query"]].to_string(index=False))

# ── Step 3: Similarity filter ────────────────────────────────────────
sim_threshold = 0.3
df_hits = df[df["similarity_to_query"] >= sim_threshold].copy()
print(f"\nStep 2 — Similarity filter (Tanimoto >= {sim_threshold}): {len(df_hits)} / {len(df)} pass")

In [ ]:
# ── Step 4: Drug-likeness filter (Lipinski's Rule of Five) ────────────
def passes_lipinski(mol):
    """Returns True if molecule passes all four Lipinski criteria."""
    return (
        Descriptors.MolWt(mol) <= 500
        and Descriptors.MolLogP(mol) <= 5
        and rdMolDescriptors.CalcNumHBD(mol) <= 5
        and rdMolDescriptors.CalcNumHBA(mol) <= 10
    )

df_hits["lipinski"] = df_hits["mol"].apply(passes_lipinski)

# Also add the property values for inspection
df_hits["MW"] = df_hits["mol"].apply(Descriptors.MolWt)
df_hits["LogP"] = df_hits["mol"].apply(Descriptors.MolLogP)
df_hits["HBD"] = df_hits["mol"].apply(rdMolDescriptors.CalcNumHBD)
df_hits["HBA"] = df_hits["mol"].apply(rdMolDescriptors.CalcNumHBA)

print("Step 3 — Lipinski filter:")
print(df_hits[["Name", "similarity_to_query", "MW", "LogP", "HBD", "HBA", "lipinski"]].to_string(index=False))

df_filtered = df_hits[df_hits["lipinski"]].copy()
print(f"\n  {len(df_filtered)} / {len(df_hits)} hits pass Lipinski")

In [ ]:
# ── Step 5: Diversity pick from filtered hits ─────────────────────────
if len(df_filtered) > 5:
    filtered_fps = [morgan_gen.GetFingerprint(m) for m in df_filtered["mol"]]
    pick_ids = list(picker.LazyBitVectorPick(filtered_fps, len(filtered_fps), min(5, len(filtered_fps))))
    df_diverse = df_filtered.iloc[pick_ids].copy()
else:
    df_diverse = df_filtered.copy()

print(f"Step 4 — Diversity pick: {len(df_diverse)} compounds selected")
print()

# ── Pipeline summary ──────────────────────────────────────────────────
print("=" * 60)
print("VIRTUAL SCREENING PIPELINE SUMMARY")
print("=" * 60)
print(f"  Starting library:           {len(df):>4} compounds")
print(f"  After similarity filter:    {len(df_hits):>4} compounds  (Tc >= {sim_threshold})")
print(f"  After Lipinski filter:      {len(df_filtered):>4} compounds")
print(f"  After diversity pick:       {len(df_diverse):>4} compounds")
print("=" * 60)
print()
print("Final selections:")
print(df_diverse[["Name", "similarity_to_query"]].to_string(index=False))

In [ ]:
# ── Visualize the screening funnel ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Funnel bar chart
stages = ["Full Library", f"Similarity >= {sim_threshold}", "Lipinski Pass", "Diversity Pick"]
counts = [len(df), len(df_hits), len(df_filtered), len(df_diverse)]
bar_colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]

axes[0].barh(stages[::-1], counts[::-1], color=bar_colors[::-1], edgecolor="black", linewidth=0.5)
for i, (stage, count) in enumerate(zip(stages[::-1], counts[::-1])):
    axes[0].text(count + 0.3, i, str(count), va="center", fontweight="bold")
axes[0].set_xlabel("Number of Compounds")
axes[0].set_title("Screening Funnel", fontsize=13)

# Right: Final picks on PCA
diverse_indices = df_diverse.index.tolist()
diverse_set = set(diverse_indices)
hit_set = set(df_hits.index.tolist())

point_colors = []
point_sizes = []
for i in range(len(df)):
    if i in diverse_set:
        point_colors.append("#e74c3c")  # red = final pick
        point_sizes.append(120)
    elif i in hit_set:
        point_colors.append("#f39c12")  # orange = hit but not picked
        point_sizes.append(60)
    else:
        point_colors.append("#bdc3c7")  # gray = filtered out
        point_sizes.append(30)

axes[1].scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    c=point_colors, s=point_sizes, alpha=0.85,
    edgecolors="black", linewidth=0.5
)
for i in diverse_indices:
    axes[1].annotate(
        df.iloc[i]["Name"], (pca_coords[i, 0], pca_coords[i, 1]),
        fontsize=7, fontweight="bold", ha="center", va="bottom",
        xytext=(0, 7), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", fc="yellow", alpha=0.6)
    )
axes[1].set_title("Pipeline Output on Chemical Space", fontsize=13)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

# Manual legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Final picks'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=8, label='Hits (not picked)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#bdc3c7', markersize=6, label='Filtered out'),
]
axes[1].legend(handles=legend_elements, loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

## 7. R-Group Decomposition

### Chemistry refresher: Computational SAR Tables

R-group decomposition is the computational version of what medicinal chemists do instinctively when building SAR tables. Given a set of molecules and a common core (scaffold), it breaks each molecule into:
- **Core** — the shared framework
- **R1, R2, ...** — the varying substituents at defined positions

This is essential for:
- **Understanding SAR** — which R-groups drive activity?
- **Identifying optimization vectors** — where can you modify the molecule?
- **Communicating results** — SAR tables are the lingua franca of med chem

> **Wet-lab bridge:** You have drawn these tables countless times on whiteboards and in presentations. RDKit's `rdRGroupDecomposition` automates the tedious part — aligning molecules to a core and extracting substituents. The interpretation (which R-groups matter) still requires your chemical intuition.

We will try to decompose our kinase inhibitors using a pyrimidine-amine core — a common hinge-binding motif in Type I kinase inhibitors.

In [ ]:
# ── R-Group Decomposition ─────────────────────────────────────────────
# Try a quinazoline/pyrimidine core (shared by EGFR inhibitors)
core_smarts = "c1cnc2cccc(N)c2n1"  # aminopyrimidine-type hinge binder
core = Chem.MolFromSmarts(core_smarts)

# Find molecules matching this core
matching_mols = []
matching_names = []
for _, row in df.iterrows():
    if row["mol"].HasSubstructMatch(core):
        matching_mols.append(row["mol"])
        matching_names.append(row["Name"])

print(f"Core SMARTS: {core_smarts}")
print(f"Molecules matching core: {len(matching_mols)}")
if matching_names:
    print(f"  {', '.join(matching_names)}")

# If we got matches, decompose
if len(matching_mols) >= 2:
    res, unmatched = rdRGroupDecomposition.RGroupDecompose(
        [core], matching_mols, asSmiles=True
    )
    if res:
        df_rgroups = pd.DataFrame(res)
        df_rgroups.insert(0, "Name", matching_names[:len(df_rgroups)])
        print(f"\nR-Group Decomposition ({len(df_rgroups)} molecules):")
        print(df_rgroups.to_string(index=False))
    else:
        print("  Decomposition returned no results with this core.")
else:
    print("\nNot enough matches. Trying a broader core...")

In [ ]:
# ── Fallback: use the quinazoline core (EGFR inhibitor series) ────────
# Gefitinib, Erlotinib, Lapatinib, Afatinib, Osimertinib share a 
# 4-anilinoquinazoline core
quinazoline_core = Chem.MolFromSmarts("[#7]c1ncnc2ccccc12")  # 4-aminoquinazoline

matching_mols2 = []
matching_names2 = []
for _, row in df.iterrows():
    if row["mol"].HasSubstructMatch(quinazoline_core):
        matching_mols2.append(row["mol"])
        matching_names2.append(row["Name"])

print(f"4-Aminoquinazoline core matches: {matching_names2}")

if len(matching_mols2) >= 2:
    res2, unmatched2 = rdRGroupDecomposition.RGroupDecompose(
        [quinazoline_core], matching_mols2, asSmiles=True
    )
    if res2:
        df_rg2 = pd.DataFrame(res2)
        df_rg2.insert(0, "Name", matching_names2[:len(df_rg2)])
        print(f"\nR-Group Decomposition on 4-aminoquinazoline core:")
        print(df_rg2.to_string(index=False))
    else:
        print("  No decomposition results.")

# Also try urea-containing compounds (Sorafenib, Regorafenib)
urea_core = Chem.MolFromSmarts("c1ccc(NC(=O)Nc2ccccc2)cc1")  # diaryl urea
matching_mols3 = []
matching_names3 = []
for _, row in df.iterrows():
    if row["mol"].HasSubstructMatch(urea_core):
        matching_mols3.append(row["mol"])
        matching_names3.append(row["Name"])

if len(matching_mols3) >= 2:
    print(f"\nDiaryl urea core matches: {matching_names3}")
    res3, _ = rdRGroupDecomposition.RGroupDecompose(
        [urea_core], matching_mols3, asSmiles=True
    )
    if res3:
        df_rg3 = pd.DataFrame(res3)
        df_rg3.insert(0, "Name", matching_names3[:len(df_rg3)])
        print(f"\nR-Group Decomposition on diaryl urea core:")
        print(df_rg3.to_string(index=False))

## 8. Mini Case Study: Full Pipeline Summary

Let us pull everything together and review our kinase inhibitor library from compound loading through to final diversity selection. This mirrors how a computational chemist would document a screening campaign.

### Pipeline Recap

| Step | What We Did | Key Method |
|------|------------|------------|
| 1. Load library | Parsed 29 kinase inhibitor SMILES | `Chem.MolFromSmiles()` |
| 2. Scaffold analysis | Extracted Murcko frameworks | `MurckoScaffold.GetScaffoldForMol()` |
| 3. Chemical space | Projected fingerprints to 2D | PCA + t-SNE on Morgan FPs |
| 4. Clustering | Grouped by structural similarity | Butina + KMeans |
| 5. Virtual screening | Ranked by similarity to Imatinib | Tanimoto on Morgan FPs |
| 6. Property filter | Applied Lipinski's Rule of Five | `Descriptors.MolWt()`, etc. |
| 7. Diversity pick | Selected diverse representatives | `MaxMinPicker` |
| 8. R-group decomposition | Extracted SAR tables for subseries | `rdRGroupDecomposition` |

> **Key takeaway:** Each step is simple on its own (you learned them in earlier notebooks). The power comes from chaining them into a coherent workflow. The order matters: filter aggressively early (cheap computation) to reduce the set before expensive analyses.

In [ ]:
# ── Final results table ───────────────────────────────────────────────
# Build a comprehensive summary for each molecule
summary_cols = ["Name", "scaffold_smi", "butina_cluster", "kmeans_cluster", "similarity_to_query"]
df_summary = df[summary_cols].copy()
df_summary["MW"] = df["mol"].apply(lambda m: round(Descriptors.MolWt(m), 1))
df_summary["LogP"] = df["mol"].apply(lambda m: round(Descriptors.MolLogP(m), 2))
df_summary["HBD"] = df["mol"].apply(rdMolDescriptors.CalcNumHBD)
df_summary["HBA"] = df["mol"].apply(rdMolDescriptors.CalcNumHBA)
df_summary["Rings"] = df["mol"].apply(rdMolDescriptors.CalcNumRings)
df_summary["RotBonds"] = df["mol"].apply(rdMolDescriptors.CalcNumRotatableBonds)

# Mark which ones were in the final diversity pick
df_summary["Selected"] = df_summary.index.isin(df_diverse.index)

print("Complete Library Summary:")
print("=" * 120)
display_cols = ["Name", "MW", "LogP", "HBD", "HBA", "Rings", "RotBonds", 
                "butina_cluster", "kmeans_cluster", "similarity_to_query", "Selected"]
print(df_summary[display_cols].to_string(index=False))
print()
print(f"Selected compounds are marked True — these are the final picks from our pipeline.")

In [ ]:
# ── Show final selected compounds as a grid ───────────────────────────
selected_mols = df_diverse["mol"].tolist()
selected_names = df_diverse["Name"].tolist()
selected_sims = [f"{s:.2f}" for s in df_diverse["similarity_to_query"]]
legends = [f"{n}\n(Tc={s})" for n, s in zip(selected_names, selected_sims)]

print(f"Final diverse picks from virtual screening pipeline ({len(selected_mols)} compounds):")
Draw.MolsToGridImage(
    selected_mols,
    molsPerRow=min(3, len(selected_mols)),
    subImgSize=(400, 300),
    legends=legends,
)

In [ ]:
# ── Property distribution of the full library ─────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

props = [
    ("MW", "Molecular Weight (Da)", 500),
    ("LogP", "cLogP", 5),
    ("HBD", "H-Bond Donors", 5),
    ("HBA", "H-Bond Acceptors", 10),
]

for ax, (col, label, limit) in zip(axes.flat, props):
    ax.hist(df_summary[col], bins=12, color="steelblue", edgecolor="black", alpha=0.7)
    ax.axvline(limit, color="red", linestyle="--", linewidth=2, label=f"Lipinski limit = {limit}")
    
    # Highlight selected compounds
    selected_vals = df_summary[df_summary["Selected"]][col]
    for val in selected_vals:
        ax.axvline(val, color="orange", linestyle="-", linewidth=1, alpha=0.5)
    
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

fig.suptitle("Property Distributions (red = Lipinski limit, orange = selected compounds)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 9. Summary and Key Takeaways

### What We Built

This notebook demonstrated the **complete computational workflow** that underpins modern drug discovery campaigns:

1. **Library Curation** — Loaded and validated a compound set (always check for parsing failures)
2. **Scaffold Analysis** — Extracted Murcko frameworks to understand structural diversity at the core level
3. **Chemical Space Visualization** — Used PCA and t-SNE to create 2D maps of fingerprint space
4. **Clustering** — Applied Butina (chemistry-native, distance-based) and KMeans (general-purpose, partition-based) to group related compounds
5. **Diversity Picking** — Used MaxMinPicker to select maximally diverse subsets for resource-constrained follow-up
6. **Virtual Screening** — Built an end-to-end pipeline: query -> similarity search -> property filter -> diversity pick
7. **R-Group Decomposition** — Extracted computational SAR tables for compound subseries

### Design Principles

These workflows follow a consistent pattern:

```
Load  ->  Characterize  ->  Filter  ->  Prioritize  ->  Select
```

At each stage, you reduce the set size while increasing the information per compound. This is the **screening funnel** — wide at the top (millions of compounds), narrow at the bottom (tens of candidates).

### Where These Fit in Drug Discovery

| Stage | Scale | What Happens | Notebooks |
|-------|-------|-------------|-----------|
| Library design | 10^6 compounds | Diversity analysis, property profiling | This notebook |
| Virtual screening | 10^5 - 10^6 | Similarity search, pharmacophore matching | This notebook |
| Hit triage | 10^2 - 10^3 | Clustering, filtering, cherry-picking | This notebook |
| Lead optimization | 10^1 - 10^2 | R-group analysis, SAR tables | This notebook |
| ADMET profiling | 10^1 | Property prediction, alerts | Notebooks 6-8 |

### What is Next

**Notebook 11** will provide exercises to practice these workflows on different datasets and scenarios — including building your own end-to-end pipelines from scratch.

---

*These techniques form the computational backbone of modern drug discovery. The tools are mature, the workflows are well-established, and the skills transfer directly to industry positions in computational chemistry, cheminformatics, and data science for drug discovery.*